CoPilot cannot generate image descriptions using just the IIIF manifest URLs; instead, it requires images to be embedded in the spreadsheet.

In [5]:
import pandas as pd
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter
import requests
from io import BytesIO
from PIL import Image as PILImage
import os

# Load the consolidated TSV
consolidated_file = '/Users/maeve/Documents/DAIMS-search/consolidated_exhibition_data.tsv'
df = pd.read_csv(consolidated_file, sep='\t', encoding='latin-1')

# Display columns and sample data
print("Columns:", df.columns.tolist())
print("\nDataset keys:", df['dataset_key'].unique() if 'dataset_key' in df.columns else "N/A")
print("\nFirst few rows:")
print(df.head())

Columns: ['manifest', 'record_url', 'dataset_key', 'dataset_name', 'geometry_type', 'coordinates', 'thumbnail']

Dataset keys: <StringArray>
['handmade', 'custom_iiif', 'comp1']
Length: 3, dtype: str

First few rows:
                                            manifest  \
0  https://iiif.library.leeds.ac.uk/presentation/...   
1  https://iiif.library.leeds.ac.uk/presentation/...   
2  https://iiif.library.leeds.ac.uk/presentation/...   
3  https://iiif.library.leeds.ac.uk/presentation/...   
4  https://iiif.library.leeds.ac.uk/presentation/...   

                                   record_url dataset_key  \
0  https://digital.library.leeds.ac.uk/15933/    handmade   
1  https://digital.library.leeds.ac.uk/15941/    handmade   
2  https://digital.library.leeds.ac.uk/15944/    handmade   
3  https://digital.library.leeds.ac.uk/15930/    handmade   
4  https://digital.library.leeds.ac.uk/15925/    handmade   

             dataset_name geometry_type  \
0  Hand-mapped Leeds Data       Poly

In [6]:
# Filter for hand-mapped data (dataset_key == 'handmade')
hand_mapped = df[df['dataset_key'] == 'handmade'].copy()
hand_mapped.columns

Index(['manifest', 'record_url', 'dataset_key', 'dataset_name',
       'geometry_type', 'coordinates', 'thumbnail'],
      dtype='str')

In [8]:
def download_and_resize_image(url, max_width=150, max_height=200):
    """Download image from URL and resize for Excel embedding"""
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            img = PILImage.open(BytesIO(response.content))
            
            # Resize while maintaining aspect ratio
            img.thumbnail((max_width, max_height), PILImage.Resampling.LANCZOS)
            
            # Save to BytesIO for Excel embedding
            img_io = BytesIO()
            img.save(img_io, format='PNG')
            img_io.seek(0)
            return img_io
    except Exception as e:
        print(f"  Error downloading {url}: {e}")
    return None

# Create Excel workbook
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Hand-Mapped Items"

# Get all columns from hand_mapped and add 'Image' column at the beginning
all_columns = ['image'] + hand_mapped.columns.tolist()
ws.append(all_columns)

# Set column widths
ws.column_dimensions['A'].width = 20  # Image column
for col_idx in range(2, len(all_columns) + 1):
    col_letter = get_column_letter(col_idx)
    ws.column_dimensions[col_letter].width = 15

# Populate rows with data and images
print("Embedding images in Excel")
for idx, (_, row) in enumerate(hand_mapped.iterrows(), start=2):
    ws.row_dimensions[idx].height = 150  # Height for images
    
    # Add all data columns (starting from column B)
    for col_idx, col_name in enumerate(hand_mapped.columns, start=2):
        col_letter = get_column_letter(col_idx)
        ws[f'{col_letter}{idx}'] = row[col_name]
    
    # Download and embed image in column A
    thumbnail_url = row.get('thumbnail')
    if thumbnail_url and pd.notna(thumbnail_url):
        print(f"  Row {idx}: Downloading {str(thumbnail_url)[:50]}...")
        img_io = download_and_resize_image(thumbnail_url)
        if img_io:
            try:
                xl_img = XLImage(img_io)
                ws.add_image(xl_img, f'A{idx}')
                print(f"image embedded")
            except Exception as e:
                print(f"failed to embed: {e}")
        else:
            print(f"image unavailable")
    else:
        print(f"  Row {idx}: No thumbnail URL")

# Save the workbook
output_path = '/Users/maeve/Documents/DAIMS-search/handmade_exhibition_with_images.xlsx'
wb.save(output_path)


Embedding images in Excel
  Row 2: Downloading https://iiif.library.leeds.ac.uk/thumbs/j5rkp6fs_o...
image embedded
  Row 3: Downloading https://iiif.library.leeds.ac.uk/thumbs/hj66rxgy_o...
image embedded
  Row 4: Downloading https://iiif.library.leeds.ac.uk/thumbs/n3yzz5xc_o...
image embedded
  Row 5: Downloading https://iiif.library.leeds.ac.uk/thumbs/x181hghv_o...
image embedded
  Row 6: Downloading https://iiif.library.leeds.ac.uk/thumbs/r5r7k2ft_o...
image embedded
  Row 7: Downloading https://iiif.library.leeds.ac.uk/thumbs/ndwpm1r1_o...
image embedded
  Row 8: Downloading https://iiif.library.leeds.ac.uk/thumbs/vc3bghvk_o...
image embedded
  Row 9: Downloading https://iiif.library.leeds.ac.uk/thumbs/gmhx8739_o...
image embedded
  Row 10: Downloading https://iiif.library.leeds.ac.uk/thumbs/j6v71hl9_o...
image embedded
  Row 11: Downloading https://iiif.library.leeds.ac.uk/thumbs/rwp4hcxz_o...
image embedded
  Row 12: Downloading https://iiif.library.leeds.ac.uk/thumbs/th197jpj_o